# Section-Aware Chunker — KDIGO + NICE

Turns parsed pages into retrieval-ready chunks with full citation metadata.

**Inputs**
- `corpus/parsed/kdigo_parsed.json` (166 pages, 54 sections)
- `corpus/parsed/nice_parsed.json` (74 pages, 16 sections)

**Output**
- `corpus/chunks/all_chunks.jsonl` — one JSON object per line

**Chunking spec (per instructor)**
- Target chunk size: **500 tokens** (cl100k_base)
- Overlap: **50 tokens (10%)** — only applied when a section splits
- Target total: **≤500 chunks**
- Boundary rule: never cut inside a `Recommendation X.X.X:` or `Practice Point X.X.X:` block; each chunk stays within one `section_title`.

**Metadata per chunk**
```json
{"chunk_id", "document_name", "source_url", "page_number",
 "page_range", "section_title", "token_count", "text"}
```

In [1]:
import json
import os
import re
import tiktoken

PARSED_DIR = "corpus/parsed"
OUT_DIR = "corpus/chunks"
os.makedirs(OUT_DIR, exist_ok=True)

# Chunking config
CHUNK_SIZE = 550          # target tokens per chunk (well inside deck's 400-800 range)
CHUNK_OVERLAP = 55        # 10% overlap on splits
MIN_CHUNK_SIZE = 100      # merge tiny tail chunks into previous if under this
HARD_MAX = 750            # absolute ceiling — force-split if chunk would exceed this

# Pages to exclude from indexing (duplicates or non-clinical front matter).
# KDIGO pages 34-53 are the "Summary of Recommendations" — every rec here is also
# in the body (pages 54-179). Indexing both would give Precision@K duplicate hits.
SKIP_PAGES = {
    "kdigo": set(range(34, 54)),   # summary of recs: pages 34-53 inclusive
    "nice": set(),                 # NICE has no duplicate summary section
}

# Token counter (used by embedding APIs like OpenAI; works well as a general proxy)
ENCODING = tiktoken.get_encoding("cl100k_base")


def n_tokens(text: str) -> int:
    return len(ENCODING.encode(text))


print(f"Config: chunk_size={CHUNK_SIZE} tokens, overlap={CHUNK_OVERLAP} tokens, hard_max={HARD_MAX}")
print(f"Skipping duplicate/non-clinical pages: kdigo={sorted(SKIP_PAGES['kdigo'])[:5]}...-{max(SKIP_PAGES['kdigo'])} ({len(SKIP_PAGES['kdigo'])} pages)")
print(f"Sanity check: n_tokens('Hello world') = {n_tokens('Hello world')}")

Config: chunk_size=550 tokens, overlap=55 tokens, hard_max=750
Skipping duplicate/non-clinical pages: kdigo=[34, 35, 36, 37, 38]...-53 (20 pages)
Sanity check: n_tokens('Hello world') = 2


## Step 1 — Load parsed pages

In [2]:
PARSED_FILES = [
    ("kdigo", "kdigo_parsed.json"),
    ("nice",  "nice_parsed.json"),
]

docs = {}   # short_name -> list of page dicts
for short, fname in PARSED_FILES:
    with open(os.path.join(PARSED_DIR, fname), encoding="utf-8") as f:
        pages_all = json.load(f)
    # Filter out skipped pages (duplicate summary sections)
    pages = [p for p in pages_all if p["page_number"] not in SKIP_PAGES[short]]
    skipped = len(pages_all) - len(pages)
    docs[short] = pages
    total_tokens = sum(n_tokens(p["text"]) for p in pages)
    print(f"  {short:6}: {len(pages):3} pages ({skipped} skipped) | {total_tokens:>7,} tokens | {len({p['section_title'] for p in pages})} sections")

  kdigo : 146 pages (20 skipped) | 173,379 tokens | 49 sections
  nice  :  74 pages (0 skipped) |  26,792 tokens | 16 sections


## Step 2 — Group consecutive pages by section

A single section may span multiple pages. We concatenate consecutive same-section pages into one text blob, remembering the page range so citations stay accurate.

In [3]:
def group_by_section(pages):
    """Group consecutive pages sharing the same section_title.
    Returns list of {document_name, source_url, section_title,
                     page_start, page_end, page_numbers, text}."""
    groups = []
    for p in pages:
        if groups and groups[-1]["section_title"] == p["section_title"]:
            g = groups[-1]
            g["page_end"] = p["page_number"]
            g["page_numbers"].append(p["page_number"])
            g["text"] += "\n\n" + p["text"]
        else:
            groups.append({
                "document_name": p["document_name"],
                "source_url": p["source_url"],
                "section_title": p["section_title"],
                "page_start": p["page_number"],
                "page_end": p["page_number"],
                "page_numbers": [p["page_number"]],
                "text": p["text"],
            })
    return groups


section_groups = {short: group_by_section(pages) for short, pages in docs.items()}

for short, groups in section_groups.items():
    print(f"\n{short}: {len(groups)} section groups")
    for g in groups[:5]:
        toks = n_tokens(g["text"])
        pages_str = f"p{g['page_start']}" if g['page_start']==g['page_end'] else f"p{g['page_start']}–{g['page_end']}"
        print(f"  {pages_str:>9} | {toks:>5} tok | {g['section_title'][:70]}")
    if len(groups) > 5:
        print(f"  ...({len(groups)-5} more)")


kdigo: 49 section groups
        p14 |   612 tok | notice
     p15–33 | 21413 tok | foreword
     p54–55 |  2289 tok | 1.1.1 Detection of CKD
        p56 |   997 tok | 1.1.2 Methods for staging of CKD
        p57 |  1283 tok | 1.1.3 Evaluation of chronicity
  ...(44 more)

nice: 16 section groups
         p5 |   123 tok | Overview
      p6–12 |  2609 tok | 1.1 Investigations for chronic kidney disease
     p13–14 |   915 tok | 1.2 Classification of CKD in adults
     p15–17 |  1240 tok | 1.3 Frequency of monitoring
     p18–20 |   984 tok | 1.4 Information and education for people with CKD
  ...(11 more)


## Step 3 — Split each section into 500-token chunks

**Strategy:**
1. Split section text into **atomic blocks** on `\n\n` (paragraph boundaries). Recommendations and Practice Points naturally sit in their own blocks after our parsing step.
2. Greedily pack blocks into a chunk while `sum(tokens) + next_block <= CHUNK_SIZE`.
3. On overflow: emit the current chunk, start a new one with 10% token overlap from the tail of the previous chunk.
4. If a single block is bigger than `CHUNK_SIZE`: keep it whole rather than break a recommendation. It becomes an oversized-but-coherent chunk.
5. Merge a tiny trailing chunk (< `MIN_CHUNK_SIZE` tokens) back into the previous one so we don't emit 30-token fragments.

In [4]:
SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9•\-])')


def overlap_tail(text: str, n_overlap_tokens: int) -> str:
    """Return the last ~n_overlap_tokens of text (sentence-boundary preferred)."""
    tokens = ENCODING.encode(text)
    if len(tokens) <= n_overlap_tokens:
        return text
    tail = ENCODING.decode(tokens[-n_overlap_tokens:])
    m = re.search(r'[.!?]\s+([A-Z])', tail)
    if m:
        tail = tail[m.start() + 2:]
    return tail.strip()


def split_long_block(block: str, chunk_size: int) -> list[str]:
    """Sentence-level split for a paragraph that alone exceeds chunk_size.
    Falls back to token-window split if a single sentence is still too big."""
    sentences = SENTENCE_SPLIT.split(block)
    out = []
    current, current_tokens = [], 0
    for s in sentences:
        st = n_tokens(s)
        # single sentence over cap: hard token-window split (rare)
        if st > chunk_size:
            if current:
                out.append(" ".join(current))
                current, current_tokens = [], 0
            toks = ENCODING.encode(s)
            for i in range(0, len(toks), chunk_size):
                out.append(ENCODING.decode(toks[i:i + chunk_size]))
            continue
        if current_tokens + st <= chunk_size:
            current.append(s)
            current_tokens += st
        else:
            out.append(" ".join(current))
            current, current_tokens = [s], st
    if current:
        out.append(" ".join(current))
    return out


def split_section(section_text: str,
                  chunk_size: int = CHUNK_SIZE,
                  overlap: int = CHUNK_OVERLAP,
                  hard_max: int = HARD_MAX) -> list[str]:
    """Split one section into ≤chunk_size-token pieces at paragraph → sentence → token boundaries.

    Rule: respect paragraph boundaries first (recommendations sit in their own paragraphs);
    fall back to sentence-level splitting only when a single paragraph exceeds chunk_size.
    hard_max is the absolute cap — even oversized recommendations get sentence-split at this ceiling.
    """
    blocks = [b.strip() for b in re.split(r'\n\s*\n', section_text) if b.strip()]
    if not blocks:
        return []

    # First pass: any block over hard_max gets sentence-split up front so nothing exceeds the ceiling
    normalized = []
    for b in blocks:
        if n_tokens(b) > hard_max:
            normalized.extend(split_long_block(b, chunk_size))
        else:
            normalized.append(b)

    # Second pass: pack normalized blocks into ≤chunk_size chunks with overlap
    chunks = []
    current, current_tokens = [], 0
    for block in normalized:
        block_tokens = n_tokens(block)

        if current_tokens + block_tokens <= chunk_size:
            current.append(block)
            current_tokens += block_tokens
            continue

        # emit current, start new with overlap
        if current:
            chunk_text = "\n\n".join(current)
            chunks.append(chunk_text)
            tail = overlap_tail(chunk_text, overlap)
            current = [tail, block] if tail else [block]
            current_tokens = n_tokens("\n\n".join(current))
        else:
            # first block alone bigger than chunk_size but under hard_max → keep whole
            chunks.append(block)
            current, current_tokens = [], 0

    if current:
        chunks.append("\n\n".join(current))

    # Merge tiny trailing chunk back into previous
    if len(chunks) >= 2 and n_tokens(chunks[-1]) < MIN_CHUNK_SIZE:
        chunks[-2] = chunks[-2] + "\n\n" + chunks.pop()

    return chunks


# Sanity test on one KDIGO section
test_group = section_groups["kdigo"][8]
test_chunks = split_section(test_group["text"])
print(f"Test section: {test_group['section_title']}")
print(f"  Section tokens: {n_tokens(test_group['text'])}")
print(f"  Split into {len(test_chunks)} chunks: {[n_tokens(c) for c in test_chunks]}")

Test section: 1.2.3 Guidance to clinical laboratories
  Section tokens: 2689
  Split into 7 chunks: [515, 198, 467, 335, 549, 598, 282]


## Step 4 — Build chunks + attach citation metadata

For each chunk we record:
- `page_number` — starting page (used for the human-visible citation)
- `page_range` — [start, end] if the chunk spans pages (from its section group)
- `chunk_id` — `<docshort>_p<start_page>_c<index>` (unique + human-readable)

In [5]:
def build_chunks(short_name: str, groups: list[dict]) -> list[dict]:
    out = []
    per_page_counter: dict[int, int] = {}     # page_start -> running index
    for g in groups:
        pieces = split_section(g["text"])
        for piece in pieces:
            page_start = g["page_start"]
            per_page_counter[page_start] = per_page_counter.get(page_start, 0) + 1
            idx = per_page_counter[page_start]
            out.append({
                "chunk_id": f"{short_name}_p{page_start}_c{idx:02d}",
                "document_name": g["document_name"],
                "source_url": g["source_url"],
                "page_number": page_start,
                "page_range": [g["page_start"], g["page_end"]],
                "section_title": g["section_title"],
                "token_count": n_tokens(piece),
                "text": piece,
            })
    return out


all_chunks = []
for short in docs.keys():
    doc_chunks = build_chunks(short, section_groups[short])
    all_chunks.extend(doc_chunks)
    print(f"  {short}: {len(doc_chunks)} chunks")

print(f"\nTOTAL: {len(all_chunks)} chunks")
assert len(all_chunks) <= 500, f"Instructor target ≤500 chunks, got {len(all_chunks)}"
print("✓ Under instructor target of 500 chunks")

  kdigo: 405 chunks
  nice: 72 chunks

TOTAL: 477 chunks
✓ Under instructor target of 500 chunks


## Step 5 — Save as JSONL

In [6]:
OUT_PATH = os.path.join(OUT_DIR, "all_chunks.jsonl")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

size_kb = os.path.getsize(OUT_PATH) / 1024
print(f"Saved {len(all_chunks)} chunks to {OUT_PATH} ({size_kb:.1f} KB)")

Saved 477 chunks to corpus/chunks\all_chunks.jsonl (1021.5 KB)


## Step 6 — Distribution stats

Chunks should cluster near 400–500 tokens (the target). A wide spread would suggest section boundaries dominating over the size target — that's OK for a medical corpus.

In [7]:
import statistics
sizes = [c["token_count"] for c in all_chunks]
print(f"Chunk token stats over {len(sizes)} chunks:")
print(f"  min:    {min(sizes)}")
print(f"  median: {statistics.median(sizes):.0f}")
print(f"  mean:   {statistics.mean(sizes):.0f}")
print(f"  max:    {max(sizes)}")
print(f"  total tokens: {sum(sizes):,}")

# Buckets
buckets = {
    "<200":      sum(1 for s in sizes if s < 200),
    "200–400":   sum(1 for s in sizes if 200 <= s < 400),
    "400–600":   sum(1 for s in sizes if 400 <= s <= 600),
    "600–800":   sum(1 for s in sizes if 600 < s <= 800),
    ">800 (oversized rec)": sum(1 for s in sizes if s > 800),
}
print(f"\nBucket distribution:")
for name, n in buckets.items():
    bar = "█" * (n * 40 // max(sizes.__len__(), 1))
    print(f"  {name:>22}: {n:>3}  {bar}")

# Oversized chunks (single recommendation > 500 tokens) — worth listing
big = [c for c in all_chunks if c["token_count"] > 800]
if big:
    print(f"\nOversized chunks (single rec block > 800 tokens): {len(big)}")
    for c in big:
        print(f"  {c['chunk_id']} ({c['token_count']} tok) - {c['section_title'][:70]}")

Chunk token stats over 477 chunks:
  min:    68
  median: 493
  mean:   459
  max:    745
  total tokens: 219,118

Bucket distribution:
                    <200:  22  █
                 200–400: 127  ██████████
                 400–600: 288  ████████████████████████
                 600–800:  40  ███
    >800 (oversized rec):   0  


## Step 7 — Spot-check chunks

Trace 3 chunks back to their source: verify text is coherent, metadata is right, and no recommendation was cut in half.

In [8]:
def find_chunk(pred):
    for c in all_chunks:
        if pred(c):
            return c
    return None

# 1) A KDIGO chunk containing Recommendation 3.7.1 (SGLT2i)
c = find_chunk(lambda c: "Recommendation 3.7.1" in c["text"])
print("=" * 70)
print(f"[1] {c['chunk_id']} | p{c['page_number']} | {c['token_count']} tok")
print(f"    section: {c['section_title']}")
print("-" * 70)
print(c["text"][:600], "..." if len(c["text"]) > 600 else "")

# 2) A NICE chunk containing recommendation 1.6.1 (BP targets)
c = find_chunk(lambda c: c["document_name"].startswith("NICE") and "1.6.1" in c["text"])
print("\n" + "=" * 70)
print(f"[2] {c['chunk_id']} | p{c['page_number']} | {c['token_count']} tok")
print(f"    section: {c['section_title']}")
print("-" * 70)
print(c["text"][:600], "..." if len(c["text"]) > 600 else "")

# 3) A chunk with the GFR staging table (Table 2)
c = find_chunk(lambda c: "Table 2" in c["text"] and "G3a" in c["text"])
print("\n" + "=" * 70)
print(f"[3] {c['chunk_id']} | p{c['page_number']} | {c['token_count']} tok")
print(f"    section: {c['section_title']}")
print("-" * 70)
print(c["text"][:700], "..." if len(c["text"]) > 700 else "")

[1] kdigo_p99_c01 | p99 | 544 tok
    section: 3.7 Sodium-glucose cotransporter-2 inhibitors
----------------------------------------------------------------------
eGFR <30 ml/min per 1.73 m2, compared with those who
continue.508,509 In addition, a recent individual patient level
data meta-analysis demonstrated a benefit in delaying KRT
in patients with eGFR <30 ml/min per 1.73 m2.510
3.7 Sodium-glucose cotransporter-2 inhibitors
(SGLT2i)
The Work Group concurs with the KDIGO 2022 Clinical
Practice Guideline for Diabetes Management in Chronic
Kidney Disease, which stated: “We recommend treating patients with type 2 diabetes (T2D), CKD, and an eGFR ≥20 ml/
min per 1.73 m2 with an SGLT2i (1A).”23 However, in this
guideline, we offer a more general 1A recom ...

[2] nice_p23_c02 | p23 | 482 tok
    section: 1.6 Pharmacotherapy
----------------------------------------------------------------------
NICE's guideline on hypertension in adults recommends using clinic blood pressure for 
monito

## Step 8 — Sanity: no recommendation was cut across two chunks

Every `Recommendation X.X.X:` and `Practice Point X.X.X:` string should appear exactly **once** across the whole corpus (in the chunk it belongs to). If a rec appears in multiple chunks and *isn't* just overlap tail, that's a defect.

In [9]:
REC_MARK = re.compile(r'((?:Recommendation|Practice Point)\s+\d+\.\d+(?:\.\d+)?)\s*:')

counts: dict[str, list[str]] = {}
for c in all_chunks:
    for m in REC_MARK.finditer(c["text"]):
        counts.setdefault(m.group(1), []).append(c["chunk_id"])

dupes = {k: v for k, v in counts.items() if len(v) > 1}
print(f"Distinct rec/practice-point markers found: {len(counts)}")
print(f"Appearing in >1 chunk (expected some overlap): {len(dupes)}")
if dupes:
    for k, ids in list(dupes.items())[:5]:
        print(f"  {k}: {ids}")
        # Verify the duplicate is just the overlap tail, not a real split
        for cid in ids:
            c = next(x for x in all_chunks if x["chunk_id"] == cid)
            idx = c["text"].find(k)
            snippet = c["text"][idx:idx+120].replace('\n', ' ')
            print(f"    {cid}: ...{snippet}...")

Distinct rec/practice-point markers found: 87
Appearing in >1 chunk (expected some overlap): 3
  Practice Point 1.4.2: ['kdigo_p79_c01', 'kdigo_p79_c02']
    kdigo_p79_c01: ...Practice Point 1.4.2: Where a POCT device for creatinine testing is being used, generate an estimate of GFR. Use the equ...
    kdigo_p79_c02: ...Practice Point 1.4.2: Where a POCT device for creatinine testing is being used, generate an estimate of GFR. Use the equ...
  Practice Point 2.3.2: ['kdigo_p88_c02', 'kdigo_p88_c03']
    kdigo_p88_c02: ...Practice Point 2.3.2: For mortality risk prediction to guide discussions about goals of care, use externally validated m...
    kdigo_p88_c03: ...Practice Point 2.3.2: For mortality risk prediction to guide discussions about goals of care, use externally validated m...
  Practice Point 4.1.3: ['kdigo_p131_c03', 'kdigo_p131_c04']
    kdigo_p131_c03: ...Practice Point 4.1.3: Review and limit the use of over-thecounter medicines and dietary or herbal remedies that may be 

## Summary

Chunks ready at `corpus/chunks/all_chunks.jsonl`.

**Each line is one chunk:**
```json
{"chunk_id": "kdigo_p44_c02",
 "document_name": "KDIGO 2024 CKD Guideline",
 "source_url": "https://kdigo.org/...",
 "page_number": 44,
 "page_range": [44, 45],
 "section_title": "3.7 Sodium-glucose cotransporter-2 inhibitors",
 "token_count": 483,
 "text": "Recommendation 3.7.1: We recommend treating patients with type 2 diabetes..."}
```

**Next step:** Embed each chunk's `text` with the chosen model (Gemini `text-embedding-004`) and load into ChromaDB.